<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week6_data_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** These exercises are meant to be worked through yourself – treat this the same as switching off a calculator's solver mode in an exam. It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 6 – Data Exercises: Multiple Regression

Adapted from Bekes & Kezdi, *Data Analysis for Business, Economics, and Policy* – the textbook this module follows. No output shown here to check yourself against: the point is to practise multi-variable specification, functional form, categorical variables, interactions, and (in the last question) spotting omitted-variable bias, on data you haven't seen the answer for.

Easier and/or shorter exercises are marked **[\*]**; harder and/or longer exercises are marked **[\*\*]**.

Three of these five are adapted from the textbook's own chapter exercises (Q1, Q2, Q3). Q4 is also adapted, extended with a `continent` column added to the life-expectancy data specifically for this exercise (the textbook's own version isn't restricted to it, but continent grouping is what this week's categorical/interaction content calls for). **Q5 is original** – it's built to look exactly like the kind of question you'll meet in the exam and need to reason through for your portfolio: a naive causal claim from a simple regression, and a real omitted variable to find.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

cps_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/cps_gender_pay_gap.csv"
europe_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_europe_nov2017_weekday.csv"
life_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/worldbank_lifeexp_2017.csv"
cars_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/used_cars_chicago.csv"

cps = pd.read_csv(cps_path)
europe = pd.read_csv(europe_path)
life = pd.read_csv(life_path)
life['ln_gdp'] = np.log(life['gdp_per_capita'])
cars = pd.read_csv(cars_path)

## Question 1 [\*]

Use the `cps` dataset – the same US earnings data from this week's own gender-pay-gap case study, but this time pick **one** education group (`MA`, `PhD`, or `Professional`) instead of the combined sample the workshop used.

1. Regress `ln_wage` on `female` for your chosen group.
2. Compare the gender-gap coefficient to the workshop's combined-sample result (about -0.195).
3. Try adding `age`, or an interaction between `female` and `age`, the way the workshop did for the combined sample. Does the same story hold within your chosen group?
4. **Produce a comparison table** – one column per model (the plain `female` model and your extended one(s)), in the same style as the "Results" table from the workshop's own gender-pay-gap slide.
5. What does this tell you about generalising a result estimated on a combined, pooled sample down to a specific subgroup?

> **Syntax hint – a publication-style comparison table.** The workshop's own "Results" table (Section E) puts one model per column, one variable per row, coefficients with significance stars, and N at the bottom. Build it as a small helper you can reuse for the rest of these exercises:
>
> ```python
> def stars(p):
>     return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
>
> def compare_models(models, names, variables):
>     # models: list of fitted statsmodels results
>     # names: column label for each model, e.g. ['(1)', '(2)']
>     # variables: list of (param_name, display_label) tuples, e.g. [('female', 'Female')]
>     rows = []
>     for param_name, label in variables:
>         row = {'Variable': label}
>         for name, m in zip(names, models):
>             row[name] = f"{m.params[param_name]:.3f}{stars(m.pvalues[param_name])}" if param_name in m.params.index else ''
>         rows.append(row)
>     rows.append({'Variable': 'N', **{name: str(int(m.nobs)) for name, m in zip(names, models)}})
>     rows.append({'Variable': 'R²', **{name: f'{m.rsquared:.3f}' for name, m in zip(names, models)}})
>     return pd.DataFrame(rows)
>
> print(compare_models([model_a, model_b], ['(1)', '(2)'], [('female', 'Female'), ('age', 'Age')]).to_markdown(index=False))
> ```
>
> `to_markdown()` needs the `tabulate` package – if it's not already available, run `!pip install tabulate` in a cell first. You'll reuse `compare_models()` (or your own version of it) in the rest of this notebook wherever you're asked to compare models side by side.

*Your comparison of the group-specific gap to the combined-sample result:*

## Question 2 [\*\*]

Use `europe` and pick a city.

1. Regress `log(price)` on `stars`, `rating`, and `distance`.
2. Pay attention to functional form – is a plain linear specification defensible for all three, or would any benefit from a transformation?
3. Use the residuals from your best specification to shortlist the **five most underpriced hotels**.

## Question 3 [\*\*]

Using the same city and data as Question 2, consider adding another variable in an appropriate functional form. `accommodation_type` is a natural candidate here – a Hotel, an Apartment, and a Hostel with the same stars/rating/distance may still command genuinely different prices.

1. Argue for your best specification.
2. Produce a new shortlist of five underpriced hotels.
3. Compare this shortlist to the one from Question 2 – how much did adding the variable change your conclusions?
4. **Produce a comparison table** with your Question 2 model and this new model side by side (reuse `compare_models()` from Question 1).

*How much did your shortlist change, and what does that tell you about how much the extra variable actually mattered?*

## Question 4 [\*\*]

Use `life` – the World Bank life-expectancy data, now with a `continent` column added.

1. Regress `life_exp` on `ln_gdp` separately for a few different continents. Compare the slopes.
2. Now estimate a single pooled regression with continent dummies **and** their interactions with `ln_gdp` (`life_exp ~ ln_gdp * C(continent)`).
3. Interpret the coefficients and their 95% CIs, and visualise the fitted regression lines by continent on one plot.
4. **Produce a comparison table** – one column per continent-specific regression, plus a column for the pooled interaction model, using `compare_models()` from Question 1 (or your own version of it).
5. What do you conclude? Watch out for continents with very few countries in this data – how much should you trust a slope estimated from a handful of observations?

## Question 5 [\*]

A used-car dealer looks at the `cars` data (the same Toyota Camry data from Weeks 4-5) and claims:

> "Every extra mile on the odometer reduces a Camry's value. That's a real cost to the owner, mile by mile."

They've fitted `price ~ odometer` to back this up. Note: `odometer` in this dataset is already measured in **10,000s of miles** (so a value of 5 means 50,000 miles) – keep that in mind when you interpret the coefficient.

1. Fit that regression yourself. What does the coefficient say, in plain terms?
2. Is this a safe causal claim as it stands? Think about what else might be driving both `odometer` and `price` together – there's a variable already in `cars` that's a strong candidate.
3. Refit including that variable. Does the `odometer` coefficient move, and in which direction? What does that tell you about the dealer's original framing?
4. **Produce a comparison table** with the naive model and your controlled model side by side (reuse `compare_models()` from Question 1) – this before/after table is exactly the kind of evidence you'd present in the exam or your portfolio to back up a claim like this.

This is exactly the shape of question the exam and your portfolio will ask you to reason through: a naive claim from a simple regression, and a real omitted variable to find and account for.

> **Syntax hint – looking at the data before you model it.** Each of these takes a few lines, and each one should lead to a decision about your model: which variables go in, in what form, and what you need to control for. They use generic names (`df`, outcome `y`, explanatory variables `x` and `z`, a grouping variable `g`) – swap in your own.
>
> **1. How is each variable distributed?** A long right tail (skew well above 0) is a reason to consider a log.
>
> ```python
> df[[y, x, z]].hist(bins=25, figsize=(10, 3), layout=(1, 3))
> plt.tight_layout(); plt.show()
> print(df[[y, x, z]].skew().round(2))
> ```
>
> **2. What shape does `y` take against `x`?** A LOWESS curve (Week 4) shows whether a straight line is reasonable or whether the relationship bends.
>
> ```python
> from statsmodels.nonparametric.smoothers_lowess import lowess
>
> fig, ax = plt.subplots()
> ax.scatter(df[x], df[y], s=10, alpha=0.5)
> smooth = lowess(df[y], df[x], frac=0.3)   # if y is 0/1, add it=0
> ax.plot(smooth[:, 0], smooth[:, 1], color='red')
> plt.show()
> ```
>
> **3. Average `y` within bins of `x`.** The bin-scatter from Week 3. If `y` is a 0/1 variable, each point is the share of 1s in that bin.
>
> ```python
> bins = df.groupby(pd.qcut(df[x], 10), observed=True).agg(x_mean=(x, 'mean'), y_mean=(y, 'mean'))
> plt.plot(bins['x_mean'], bins['y_mean'], 'o-'); plt.show()
> ```
>
> **4. How do the explanatory variables relate to each other?** If `x` and `z` move together and `z` could also affect `y`, leaving `z` out may bias the coefficient on `x`.
>
> ```python
> print(df[[x, z]].corr().round(2))
> plt.scatter(df[x], df[z], s=10, alpha=0.5); plt.xlabel(x); plt.ylabel(z); plt.show()
> ```
>
> **5. Is the relationship the same within groups?** Separate fitted lines for each value of `g`. Clearly different slopes suggest an interaction.
>
> ```python
> fig, ax = plt.subplots()
> for level, part in df.groupby(g):
>     m = smf.ols(f'{y} ~ {x}', data=part).fit()
>     xs = np.linspace(part[x].min(), part[x].max(), 50)
>     ax.scatter(part[x], part[y], s=10, alpha=0.4, label=f'{g} = {level}')
>     ax.plot(xs, m.params['Intercept'] + m.params[x] * xs)
> ax.legend(); plt.show()
> ```
>
> Set the names first, for example `y, x, z, g = 'price', 'odometer', 'your_variable', 'your_group'`.

*Your critique of the dealer's claim, and what changed once you controlled for the confounder:*